# Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import re
import string
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

import os
import time
import pandas as pd
import gzip
import shutil
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset  # From Hugging Face
import re
import string
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd

## Constants

In [2]:
LOAD_DATA = 'LOCAL'
LOAD_DATA = 'HUG'

DATA_DIR = './data'
FILE_NAME = 'movie_data.csv'
FILE_NAME_GZ = 'movie_data.csv.gz'

LOAD_DATA_PATH = os.path.join(DATA_DIR, FILE_NAME)
LOAD_DATA_PATH_GZ = os.path.join(DATA_DIR, FILE_NAME_GZ)

In [3]:
def load_data(source="LOCAL", test_size=0.2, random_state=42):
    """
    Load dataset from LOCAL files or Hugging Face.
    Always returns a DatasetDict with 'train' and 'test'
    and columns: ['text', 'label'].
    """

    if source == "LOCAL":
        # --- Load local file ---
        if os.path.exists(LOAD_DATA_PATH):
            df = pd.read_csv(LOAD_DATA_PATH)

        elif os.path.exists(LOAD_DATA_PATH_GZ):
            with gzip.open(LOAD_DATA_PATH_GZ, "rb") as f_in:
                with open(LOAD_DATA_PATH, "wb") as f_out:
                    shutil.copyfileobj(f_in, f_out)
            df = pd.read_csv(LOAD_DATA_PATH)

        else:
            raise FileNotFoundError(
                f"No dataset found in {FILE_DIR}. Expected {FILE_NAME} or {FILE_NAME_GZ}"
            )

        # --- Rename columns to match Hugging Face schema ---
        column_map = {
            "review": "text",
            "sentiment": "label"
        }
        df = df.rename(columns=column_map)

        # --- Validate schema ---
        required_cols = {"text", "label"}
        if not required_cols.issubset(df.columns):
            raise ValueError(f"Dataset must contain columns {required_cols}, got {df.columns}")

        # --- Split ---
        train_df, test_df = train_test_split(
            df, test_size=test_size, random_state=random_state
        )

        train_df, valid_df = train_test_split(
            train_df, test_size=max(0.25, test_size), random_state=random_state
        )

        train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
        test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
        valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))

        return DatasetDict({
            "train": train_ds,
            "test": test_ds,
            "validation": valid_ds
        })

    elif source == "HUG":
        dataset = load_dataset("imdb")
        return dataset

    else:
        raise ValueError("source must be either 'LOCAL' or 'HUG'")

In [4]:
def my_tokenizer_word(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', '')
    tokenized = text.split()
    return tokenized



In [5]:
def my_tokenizer_number(text, vocab, max_len=200):
    tokens = my_tokenizer_word(text)
    indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
    # Truncate if too long
    if len(indices) > max_len:
        indices = indices[:max_len]
    return indices

In [6]:
dataset_local = load_data(source='LOCAL')
# dataset_hug = load_data(source='HUG')

# # train_data = dataset["train"]
# dataset_local, dataset_hug

In [7]:
dataset_local

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})

In [8]:
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ────────────────────────────────────────────────
#   Build vocabulary from training data only
# ────────────────────────────────────────────────

VOCAB_SIZE = 25000      # adjust between 10k–40k depending on memory/time
MIN_FREQ = 2            # words that appear < MIN_FREQ times are <unk>

# Count all words in preprocessed training reviews
word_counts = Counter()

for example in dataset_local['train']:
    word_tokens = my_tokenizer_word(example['text'])
    word_counts.update(word_tokens)

# Create vocab dictionary
vocab = {'<pad>': 0, '<unk>': 1}
# optionally: '<bos>': 2, '<eos>': 3   (we won't use them here for simplicity)


num_of_special_char = len(vocab)
# Most common words
most_common = word_counts.most_common(min(VOCAB_SIZE, len(word_counts.most_common())) - num_of_special_char)  # reserve space for specials




offset = len(vocab)
for word, _ in most_common:
    vocab[word] = offset
    offset += 1

print(f"Vocabulary size: {len(vocab)} (including specials)")
print("Top 10 words:", most_common[:10])

Vocabulary size: 25000 (including specials)
Top 10 words: [('the', 398271), ('and', 194016), ('a', 193493), ('of', 172516), ('to', 160138), ('is', 126105), ('it', 114437), ('in', 111705), ('i', 105059), ('this', 90350)]


In [9]:
from collections import Counter

def build_vocab(dataset, text_column='text', vocab_size=25000, min_freq=2, specials=['<pad>', '<unk>']):
    """
    Build a vocabulary dictionary from a Hugging Face dataset.
    
    Args:
        dataset: Hugging Face Dataset (e.g., dataset['train'])
        text_column (str): Name of the text column in the dataset
        vocab_size (int): Maximum number of words to include in vocab (including specials)
        min_freq (int): Minimum frequency for a word to be included
        specials (list): List of special tokens (like <pad>, <unk>)
        
    Returns:
        vocab (dict): Mapping word -> index
    """
    word_counts = Counter()
    
    # Count tokens in the dataset
    for example in dataset:
        tokens = my_tokenizer_word(example[text_column])
        word_counts.update(tokens)
    
    # Initialize vocab with special tokens
    vocab = {token: idx for idx, token in enumerate(specials)}
    num_specials = len(vocab)
    
    # Get most common words respecting min_freq
    filtered_words = [(word, count) for word, count in word_counts.most_common()
                      if count >= min_freq]
    
    # Limit to vocab_size
    filtered_words = filtered_words[:vocab_size - num_specials]
    
    # Add to vocab
    offset = num_specials
    for word, _ in filtered_words:
        vocab[word] = offset
        offset += 1
    
    print(f"Vocabulary size: {len(vocab)} (including specials)")
    print("Top 10 words:", filtered_words[:10])
    
    return vocab
# Assume dataset_local['train'] exists
vocab = build_vocab(dataset_local['train'], text_column='text', vocab_size=25000, min_freq=2)


Vocabulary size: 25000 (including specials)
Top 10 words: [('the', 398271), ('and', 194016), ('a', 193493), ('of', 172516), ('to', 160138), ('is', 126105), ('it', 114437), ('in', 111705), ('i', 105059), ('this', 90350)]


In [10]:
import torch
from torch.utils.data import Dataset

class TextClassificationDataset_wasted_time(Dataset):
    def __init__(self, dataset, vocab, max_len=200):
        """
        dataset: Hugging Face Dataset format (train or test split)
        vocab: dictionary mapping word -> index
        max_len: max sequence length
        """
        self.dataset = dataset
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        text = item['text']  # or 'review' depending on your column
        label = item['label']  # or 'sentiment'

        # tokenize to numbers
        indices = my_tokenizer_number(text, self.vocab, self.max_len)

        # convert to tensor
        x = torch.tensor(indices, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        num_of_token = len(x)

        # optionally pad if shorter than max_len
        if len(x) < self.max_len:
            pad_len = self.max_len - len(x)
            pad_idx = self.vocab['<pad>']
            x = torch.cat([x, torch.full((pad_len,), pad_idx, dtype=torch.long)])

        data = {
            'tokenized_text': x,
            'label': y.float(),
            'num_of_token': num_of_token
        }
        # return x, y.float(), length_of_token
        return data



In [11]:
# new and performance improved
class TextClassificationDataset(Dataset):
    def __init__(self, dataset, vocab, max_len=200):
        """
        dataset: Hugging Face Dataset format (train or test split)
        vocab: dictionary mapping word -> index
        max_len: max sequence length
        """
        self.dataset = dataset
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        text = item['text']  # or 'review' depending on your column
        label = item['label']  # or 'sentiment'
        
        # tokenize to numbers
        indices = my_tokenizer_number(text, self.vocab, self.max_len)

        # convert to tensor
        x = torch.tensor(indices, dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        num_of_token = len(x)

        data = {
            'tokenized_text': x,
            'label': y.float(),
            'num_of_token': num_of_token
        }
        # return x, y.float(), length_of_token
        return data

# Collate function
def collate_fn(batch):
    indices = [item['tokenized_text'] for item in batch]
    labels = [item['label'] for item in batch]
    num_of_token = [item['num_of_token'] for item in batch]

    indices_padded = torch.nn.utils.rnn.pad_sequence(
        indices, batch_first=True, padding_value=0  # Assuming <pad> = 0
    )

    return {
        'tokenized_text': indices_padded,
        'num_of_token': torch.tensor(num_of_token, dtype=torch.long),
        'label': torch.tensor(labels, dtype=torch.float32)
    }

In [12]:
train_dataset = TextClassificationDataset(dataset_local['train'], vocab, max_len=200)
test_dataset  = TextClassificationDataset(dataset_local['test'], vocab, max_len=200)
valid_dataset  = TextClassificationDataset(dataset_local['validation'], vocab, max_len=200)

In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=64*2, collate_fn=collate_fn)
valid_loader  = DataLoader(valid_dataset, batch_size=64*2, collate_fn=collate_fn)

In [14]:
data = next(iter(train_loader))
data['tokenized_text'][0]

tensor([   11,     7,     4,    20,    46, 12532,  1070,  1071,  2690,     3,
           29,  1732,     9,   903,     4,   250, 13509,    35, 13181,     5,
         5154,    18,  5631,    31,    12,    59,     3,    55, 12731,     6,
         2951,     2,   889,  2771, 12676,    27,   956,     1,     2,   368,
          161,     7, 19148, 21984,    18,    12,     7,   118,     2,  2690,
         1472,   952,    40,    26,    49,    78,   616,   496,  1740,    34,
         7370,   626,  1490,     2,   316,   161,     5,     2,  4091, 16639,
            3,     8,     7,  1414,     6,   105,     2, 16639,     3,    67,
         1565,  5589,    34,  3940,     1,     1,     9,  1353,     5,  1741,
            1,     8,     7,    24,  1714,    12,     2,  1622,   821,     5,
            2,    20,  1343,  1538,    95,    56,    26,   184,  3624,  2767,
           89,     2,  1071, 14005,    28, 12129,    32,   134,  1353,     9,
           67,     1,    18,  7596,     2,   316, 13509,     7, 

In [15]:
# data = next(iter(train_loader))
# data['tokenized_text'].shape, 
data['tokenized_text'][0]#, data['num_of_token'], len(data['num_of_token'])

tensor([   11,     7,     4,    20,    46, 12532,  1070,  1071,  2690,     3,
           29,  1732,     9,   903,     4,   250, 13509,    35, 13181,     5,
         5154,    18,  5631,    31,    12,    59,     3,    55, 12731,     6,
         2951,     2,   889,  2771, 12676,    27,   956,     1,     2,   368,
          161,     7, 19148, 21984,    18,    12,     7,   118,     2,  2690,
         1472,   952,    40,    26,    49,    78,   616,   496,  1740,    34,
         7370,   626,  1490,     2,   316,   161,     5,     2,  4091, 16639,
            3,     8,     7,  1414,     6,   105,     2, 16639,     3,    67,
         1565,  5589,    34,  3940,     1,     1,     9,  1353,     5,  1741,
            1,     8,     7,    24,  1714,    12,     2,  1622,   821,     5,
            2,    20,  1343,  1538,    95,    56,    26,   184,  3624,  2767,
           89,     2,  1071, 14005,    28, 12129,    32,   134,  1353,     9,
           67,     1,    18,  7596,     2,   316, 13509,     7, 

In [16]:
data['num_of_token']

tensor([200, 129, 130, 200, 110, 113, 200, 200, 172, 200, 144, 107, 200, 200,
        150, 200, 174, 116, 197, 141, 200, 200, 200, 200, 142, 200, 200, 179,
        143, 146, 200, 133, 200, 200, 162, 172, 100, 132, 185, 102, 158, 175,
        200, 121, 127, 159, 146, 200, 200, 189, 163, 200, 171, 113, 200, 200,
         94, 184, 200, 200, 200, 197, 200,  98])

In [19]:
class BasicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3, bidirectional=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=bidirectional)
        self.lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(self.lstm_output_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, indices, lengths=None):
        embedded = self.dropout(self.embedding(indices))
        _, (hidden, _) = self.lstm(embedded)
        if self.lstm.bidirectional:
            final_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            final_hidden = hidden[-1]
        return self.fc(self.dropout(final_hidden)).squeeze(1)


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

class MyRNN(nn.Module):
    def __init__(self, vocab_size, 
                 embed_dim, 
                 hidden_dim, 
                 output_dim=1, 
                 num_layers=1, 
                 dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=embed_dim, 
            padding_idx=0  # important: <pad> token gets zero vector
        )
        
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity='tanh',        # or 'relu'
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, indices, lengths=None):
        # indices shape: [batch_size, seq_len]
        
        embedded = self.embedding(indices)          # → [batch, seq_len, embed_dim]
        embedded = self.dropout(embedded)
        
        # RNN expects: batch_first=True → [batch, seq_len, embed_dim]
        output, hidden = self.rnn(embedded)
        
        # hidden shape: [num_layers, batch, hidden_dim]
        # We take the LAST layer's final hidden state
        final_hidden = hidden[-1]                    # [batch, hidden_dim]
        
        final_hidden = self.dropout(final_hidden)
        logits = self.fc(final_hidden)               # [batch, 1]
        return logits.squeeze(1)                     # [batch]

In [21]:
from tqdm import tqdm
# def train_epoch(model, loader, optimizer, loss_fn, device):
#     model.train()
#     total_loss = 0
#     correct = 0
#     total = 0
#     train_loss_hist = []
#     train_acc_hist = []
#     valid_loss_hist = []
#     valid_acc_hist = []
    
#     for batch in tqdm(loader):
#         indices = batch['tokenized_text'].to(device)
#         labels  = batch['label'].to(device)
#         lengths = batch['num_of_token']          # we don't use packed yet
        
#         optimizer.zero_grad()
#         logits = model(indices, lengths)
#         loss = loss_fn(logits, labels)
        
#         loss.backward()
#         # Optional: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
        
#         total_loss += loss.item()
#         preds = (torch.sigmoid(logits) > 0.5).float()
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

#     train_loss, train_acc = total_loss / len(loader), correct / total
    
#     return 


# def evaluate(model, loader, loss_fn, device):
#     model.eval()
#     total_loss = 0
#     correct = 0
#     total = 0
    
#     with torch.no_grad():
#         for batch in loader:
#             indices = batch['tokenized_text'].to(device)
#             labels  = batch['label'].to(device)
#             lengths = batch['num_of_token']
            
#             logits = model(indices, lengths)
#             loss = criterion(logits, labels)
            
#             total_loss += loss.item()
#             preds = (torch.sigmoid(logits) > 0.5).float()
#             correct += (preds == labels).sum().item()
#             total += labels.size(0)
#     valid_loss, valid_acc = total_loss / len(loader), correct / total
#     return total_loss / len(loader), correct / total


def run_epoch(model, loader, optimizer, loss_fn, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader) if train else loader

    with torch.set_grad_enabled(train):
        for batch in loop:
            indices = batch['tokenized_text'].to(device)
            labels  = batch['label'].float().to(device)
            lengths = batch['num_of_token']

            if train:
                optimizer.zero_grad()

            logits = model(indices)
        
            loss = loss_fn(logits, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()

            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total

    return avg_loss, acc

In [22]:
# Hyperparameters – feel free to experiment later
EMBED_DIM   = 100
HIDDEN_DIM  = 256
NUM_LAYERS  = 1          # start with 1 (stacking many vanilla RNNs usually hurts)
DROPOUT     = 0.3
LEARNING_RATE = 0.001
N_EPOCHS    = 10          # we'll do few epochs first → RNN trains slowly

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# model = MyRNN(
#                     vocab_size=len(vocab),
#                     embed_dim=EMBED_DIM,
#                     hidden_dim=HIDDEN_DIM,
#                     output_dim=1,
#                     num_layers=NUM_LAYERS,
#                     dropout=DROPOUT
#                 ).to(device)

model = BasicLSTM(vocab_size=len(vocab), 
                  embed_dim=128, 
                  hidden_dim=256, 
                  bidirectional=True).to(device)

loss_fn = nn.BCEWithLogitsLoss()   # combines sigmoid + binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# Optional: learning rate scheduler
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

Using device: cuda


In [23]:
device

device(type='cuda')

In [24]:
print("Starting RNN training...\n")
train_loss_history = []
train_acc_history = []
val_loss_history = []
val_acc_history = []
for epoch in range(1, N_EPOCHS + 1):
    # --- Training ---
    train_loss, train_acc = run_epoch(
        model, train_loader, optimizer, loss_fn, device, train=True
    )

    # --- Validation ---
    val_loss, val_acc = run_epoch(
        model, valid_loader, optimizer, loss_fn, device, train=False
    )

    # --- Save history ---
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)
    
    # --- Print progress ---
    print(f"Epoch {epoch}/{N_EPOCHS}")
    print(f" Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f" Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")


Starting RNN training...



100%|█████████████████████████████████████| 469/469 [00:10<00:00, 46.63it/s]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, N_EPOCHS + 1)

# --- Loss ---
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs, train_loss_history, label='Train Loss')
plt.plot(epochs, val_loss_history, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over epochs')
plt.legend()

# --- Accuracy ---
plt.subplot(1,2,2)
plt.plot(epochs, train_acc_history, label='Train Acc')
plt.plot(epochs, val_acc_history, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over epochs')
plt.legend()

plt.show()

In [ ]:
next(model_rnn.parameters()).device

In [ ]:
for batch in train_loader:
    inputs = batch['tokenized_text'].to(device)
    labels = batch['label'].to(device)

    print(inputs.device, labels.device)
    break

In [25]:
# ────────────────────────────────────────────────
#   Training and Evaluation Functions
# ────────────────────────────────────────────────
from tqdm import tqdm
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch in tqdm(loader):
        indices = batch['tokenized_text'].to(device)
        lengths = batch['num_of_token'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(indices, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            indices = batch['tokenized_text'].to(device)
            lengths = batch['num_of_token'].to(device)
            labels = batch['label'].to(device)
            logits = model(indices, lengths)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

# ────────────────────────────────────────────────
#   Train All Models
# ────────────────────────────────────────────────
def train_model(model, name, n_epochs=8):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    print(f"Training {name}...")
    for epoch in range(1, n_epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, test_loader, criterion, device)
        print(f"Epoch {epoch}/{n_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    print(f"Finished {name}\n")

In [27]:
train_model(model, "LSTM")

Training LSTM...


100%|█████████████████████████████████████| 469/469 [00:10<00:00, 46.74it/s]


Epoch 1/8 | Train Loss: 0.5808 Acc: 0.7025 | Val Loss: 0.6133 Acc: 0.7007


 96%|███████████████████████████████████▋ | 452/469 [00:09<00:00, 46.61it/s]


KeyboardInterrupt: 

In [13]:
torch.randn(2,1).unsqueeze(dim=2).shape

torch.Size([2, 1, 1])

In [2]:
import torch